# LeetCode 2211 — Count Collisions on a Road bằng Python

Link đề: https://leetcode.com/problems/count-collisions-on-a-road/ · Độ khó: **Medium** · Chủ đề: String, Stack, Simulation

Skeleton LeetCode:

```python
class Solution:
    def countCollisions(self, directions: str) -> int:
        ...
```

## 1. Mục tiêu học

1. **Mô phỏng chính xác** một hệ thống chuyển động liên tục bằng bước thời gian rời rạc (nhân đôi tọa độ để mọi va chạm xảy ra tại điểm nguyên).
2. Mô phỏng hiệu quả bằng **ngăn xếp**: xử lý xe từ trái sang phải, giữ trạng thái "phía trước bên trái có vật cản đứng yên không" và số xe `R` đang "chờ" va chạm.
3. Rút ra **công thức đóng**: bỏ các `L` ở đầu và các `R` ở cuối, đáp án = số xe **đang chạy** còn lại (mỗi xe chạy cuối cùng góp đúng 1 va chạm).
4. Viết lời giải một dòng bằng `lstrip` / `rstrip` và hiểu vì sao nó đúng.
5. Kiểm thử đối chiếu mô phỏng với các cách nhanh trên **mọi** chuỗi ngắn và dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

Có `n` xe trên một con đường vô hạn, đánh số `0..n−1` từ trái sang phải, mỗi xe ở một điểm riêng. `directions[i]` là `'L'` (chạy sang trái), `'R'` (chạy sang phải) hoặc `'S'` (đứng yên). Mọi xe chạy cùng tốc độ.

Đếm va chạm:
- Hai xe **chạy ngược chiều** đâm nhau: **+2**.
- Một xe chạy đâm vào xe **đứng yên**: **+1**.

Sau va chạm, các xe liên quan **dừng hẳn** tại điểm va chạm (trở thành vật cản đứng yên). Trả về tổng số va chạm.

**Ví dụ 1:** `"RLRSLL"` → `5`: xe 0 và 1 (R↔L) +2; xe 2 đâm xe 3 (đứng yên) +1; xe 4 đâm xe 3 +1; xe 5 đâm xe 4 (đã dừng) +1.

**Ví dụ 2:** `"LLRR"` → `0`: không xe nào gặp nhau.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from itertools import product
from typing import List

examples = [("RLRSLL", 5), ("LLRR", 0)]
for d, exp in examples:
    print(d, "-> kỳ vọng", exp)

RLRSLL -> kỳ vọng 5
LLRR -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Nhân đôi tọa độ để va chạm rơi vào điểm nguyên

Xe `i` ở vị trí `i`. Hai xe `R` (ở `x`) và `L` (ở `x+1`) gặp nhau tại `x + 0,5` sau `0,5` đơn vị thời gian. Nếu nhân đôi tọa độ (`2i`) và cho mỗi "nửa bước" xe chạy 1 đơn vị, thì mọi lần gặp nhau đều xảy ra tại **cùng một điểm nguyên** sau một số nguyên nửa bước — không có chuyện hai xe "lướt qua nhau".

In [2]:
pos = {0: 0, 1: 2}                 # xe 0 (R) ở 0, xe 1 (L) ở 2 (tọa độ nhân đôi)
pos[0] += 1; pos[1] -= 1
print("sau một nửa bước:", pos, "-> cùng điểm 1 -> va chạm")

sau một nửa bước: {0: 1, 1: 1} -> cùng điểm 1 -> va chạm


### 3.2 Ngăn xếp / bộ đếm "xe R đang chờ"

Duyệt từ trái sang phải, các xe `R` liên tiếp chưa va chạm "dồn" lại. Khi gặp một vật cản (`S` hoặc `L`), **tất cả** các `R` đang chờ sẽ lần lượt đâm vào (mỗi chiếc +1, riêng lần R↔L đầu tiên là +2). Chỉ cần một bộ đếm `pending_R` thay cho stack thật.

In [3]:
pending = 0
for ch in "RRRS":
    if ch == "R": pending += 1
    else: print(f"gặp {ch}: {pending} xe R đâm vào -> +{pending}"); pending = 0

gặp S: 3 xe R đâm vào -> +3


### 3.3 `lstrip` / `rstrip` với tập ký tự

`s.lstrip("L")` bỏ **mọi** ký tự `'L'` ở **đầu** chuỗi; `s.rstrip("R")` bỏ mọi `'R'` ở **cuối**. Không ảnh hưởng ký tự ở giữa.

In [4]:
s = "LLRSLRR"
print(repr(s.lstrip("L")), repr(s.rstrip("R")), repr(s.lstrip("L").rstrip("R")))

'RSLRR' 'LLRSL' 'RSL'


### 3.4 `str.count`

`t.count("S")` đếm số lần xuất hiện. Số xe đang chạy trong `t` là `len(t) − t.count("S")`.

In [5]:
t = "RSL"
print(len(t) - t.count("S"))

2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `"RLRSLL"`: vẽ hướng đi:

```
xe:   0  1  2  3  4  5
hướng: →  ←  →  ■  ←  ←
```

- Xe 0 → và xe 1 ← gặp nhau: **+2**, cả hai dừng (thành vật cản).
- Xe 2 → đâm xe 3 ■: **+1**.
- Xe 4 ← đâm xe 3 ■: **+1**, dừng.
- Xe 5 ← đâm xe 4 (đã dừng): **+1**.

Tổng `5`. Quan sát quan trọng: **mỗi xe chạy mà cuối cùng bị dừng đóng góp đúng 1** vào tổng (va chạm ngược chiều +2 = 1 cho mỗi xe trong hai xe). Xe nào **không bao giờ** dừng? Các xe `L` ở **đầu** (chạy thoát sang trái, không có gì cản) và các xe `R` ở **cuối** (thoát sang phải). Mọi xe chạy khác đều sẽ gặp vật cản.

Ví dụ 1: không có `L` ở đầu, không có `R` ở cuối ⇒ 5 xe chạy (R, L, R, L, L) ⇒ đáp án 5 ✓.

In [6]:
d = "RLRSLL"
core = d.lstrip("L").rstrip("R")
print(f"bỏ L ở đầu và R ở cuối: {d!r} -> {core!r}; số xe chạy còn lại = {len(core) - core.count('S')}")

bỏ L ở đầu và R ở cuối: 'RLRSLL' -> 'RLRSLL'; số xe chạy còn lại = 5


## 5. Cách 1 — Brute Force / baseline: mô phỏng theo nửa bước thời gian

**Ý tưởng:** tọa độ nhân đôi `2i`. Mỗi nửa bước:
1. Mọi xe đang chạy di chuyển ±1.
2. Gom xe theo vị trí. Tại mỗi vị trí có ≥ 2 xe:
   - Nếu có xe **đứng yên** ở đó (trước khi các xe chạy tới): mỗi xe chạy vừa tới **+1**.
   - Nếu không có xe đứng yên và có cả xe chạy `R` lẫn `L` tới: **+2** (một cặp ngược chiều).
   - Mọi xe tại điểm đó trở thành đứng yên.
3. Dừng khi không còn xe nào có thể gặp nhau (sau tối đa `2n` nửa bước).

In [7]:
def collisions_simulate(directions):
    n = len(directions)
    pos = [2 * i for i in range(n)]
    state = list(directions)                  # 'L', 'R', 'S'
    total = 0
    for _ in range(2 * n + 2):
        stopped_at = defaultdict(int)
        for i in range(n):
            if state[i] == "S":
                stopped_at[pos[i]] += 1
        for i in range(n):
            if state[i] == "R": pos[i] += 1
            elif state[i] == "L": pos[i] -= 1
        groups = defaultdict(list)
        for i in range(n):
            groups[pos[i]].append(i)
        for p, ids in groups.items():
            movers = [i for i in ids if state[i] != "S"]
            if not movers or len(ids) < 2:
                continue
            if stopped_at[p]:
                total += len(movers)              # mỗi xe chạy đâm vào vật cản đứng yên
            else:
                dirs = Counter(state[i] for i in movers)
                if dirs["R"] and dirs["L"]:
                    total += 2                    # một cặp ngược chiều
            for i in ids:
                state[i] = "S"
    return total

for d, exp in examples:
    got = collisions_simulate(d); print(d, "->", got); assert got == exp

RLRSLL -> 5
LLRR -> 0


**Độ phức tạp:** `O(n)` nửa bước × `O(n)` mỗi bước ⇒ `O(n²)`. Với `n = 10⁵` là `10¹⁰` — chậm. Nhưng đây là **chuẩn tin cậy** vì làm đúng định nghĩa vật lý.

**Nút thắt:** phần lớn thời gian xe chỉ "chạy không"; mọi va chạm có thể xác định chỉ bằng **thứ tự** các xe, không cần thời gian.

In [8]:
rng = random.Random(0)
for n in (200, 400, 800):
    d = "".join(rng.choice("LRS") for _ in range(n))
    t = time.perf_counter(); collisions_simulate(d)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=200: 0.016s
n=400: 0.065s


n=800: 0.264s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (quét trái → phải):** giữ hai thông tin: `blocked` = bên trái xe hiện tại đã có **vật cản đứng yên** chưa (do `S` hoặc do va chạm trước đó), và `pending_R` = số xe `R` liên tiếp chưa va chạm.
- Gặp `R`: `pending_R += 1`.
- Gặp `S`: tất cả `pending_R` xe đâm vào ⇒ `+pending_R`; `pending_R = 0`; `blocked = True`.
- Gặp `L`:
  - **nếu** `pending_R > 0` **thì** xe `R` gần nhất và `L` đâm nhau (+2), các `R` còn lại đâm vào đống đổ nát (+1 mỗi chiếc) ⇒ `+pending_R + 1`; `pending_R = 0`; `blocked = True`;
  - **nếu không** mà `blocked` **thì** `L` đâm vào vật cản ⇒ `+1`;
  - **nếu không** (không có gì bên trái) ⇒ `L` thoát, `+0`.

**Insight 2 (công thức):** mỗi xe chạy bị dừng đóng góp đúng **1** (va chạm ngược chiều +2 = 1 + 1). Xe chạy **thoát** được chỉ là các `L` ở đầu và các `R` ở cuối. ⇒ đáp án = số ký tự khác `'S'` trong `directions.lstrip('L').rstrip('R')`.

**Invariant (quét):** sau khi xử lý xe `i`, tổng đã cộng = số va chạm mà mọi xe trong `0..i` gây ra **với nhau và với vật cản bên trái**, trừ các `R` trong `pending_R` (chưa biết số phận).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quét với ngăn xếp thật

Mô phỏng từng xe với ngăn xếp các trạng thái xe bên trái (chỉ cần đỉnh): đẩy `R`; khi gặp `S` hoặc `L`, lần lượt pop các `R` và cộng điểm. Viết tường minh để dễ theo dõi.

In [9]:
def collisions_stack(directions):
    stack, total = [], 0
    for ch in directions:
        if ch == "R":
            stack.append("R")
            continue
        if ch == "L":
            if not stack:
                continue                       # thoát sang trái
            if stack[-1] == "R":
                total += 2; stack.pop()        # R <-> L
                ch = "S"                       # đống đổ nát đứng yên
            else:                              # đỉnh là S
                total += 1; ch = "S"
        # ch == 'S': các R đang chờ đâm vào
        while stack and stack[-1] == "R":
            stack.pop(); total += 1
        stack.append("S")
    return total

for d, exp in examples:
    assert collisions_stack(d) == exp
print([collisions_stack(d) for d, _ in examples])

[5, 0]


**Độ phức tạp Cách 2:** mỗi xe push/pop tối đa một lần ⇒ `O(n)`, Space `O(n)`.

### 7.2 Cách 3 — Quét với hai biến `blocked`, `pending_R`

In [10]:
def collisions_counter(directions):
    total = pending_R = 0
    blocked = False
    for ch in directions:
        if ch == "R":
            pending_R += 1
        elif ch == "S":
            total += pending_R; pending_R = 0; blocked = True
        else:  # 'L'
            if pending_R:
                total += pending_R + 1; pending_R = 0; blocked = True
            elif blocked:
                total += 1
    return total

for d, exp in examples:
    assert collisions_counter(d) == exp
print([collisions_counter(d) for d, _ in examples])

[5, 0]


In [11]:
class SolutionCounter:
    def countCollisions(self, directions: str) -> int:
        total = pending = 0
        blocked = False
        for ch in directions:
            if ch == "R":
                pending += 1
            elif ch == "S":
                total += pending; pending = 0; blocked = True
            elif pending:
                total += pending + 1; pending = 0; blocked = True
            elif blocked:
                total += 1
        return total

print([SolutionCounter().countCollisions(d) for d, _ in examples])

[5, 0]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Công thức `lstrip` / `rstrip` (bản nộp chính)

In [12]:
def collisions_strip(directions):
    core = directions.lstrip("L").rstrip("R")
    return len(core) - core.count("S")

for d, exp in examples:
    assert collisions_strip(d) == exp
print([collisions_strip(d) for d, _ in examples])

[5, 0]


**Độ phức tạp Cách 4:** `O(n)` (các hàm chuỗi chạy trong C), Space `O(n)` cho chuỗi con.

### 7.4 Cách 5 — Công thức bằng hai con trỏ (không tạo chuỗi mới)

Tìm `l` = vị trí đầu tiên không phải `'L'`, `r` = vị trí cuối cùng không phải `'R'`; đếm ký tự khác `'S'` trong `[l, r]`.

In [13]:
def collisions_two_pointers(directions):
    n = len(directions); l, r = 0, n - 1
    while l < n and directions[l] == "L": l += 1
    while r >= 0 and directions[r] == "R": r -= 1
    return sum(1 for i in range(l, r + 1) if directions[i] != "S")

for d, exp in examples:
    assert collisions_two_pointers(d) == exp
print([collisions_two_pointers(d) for d, _ in examples], "| toàn L:", collisions_two_pointers("LLL"))

[5, 0] | toàn L: 0


**Độ phức tạp Cách 5:** `O(n)`, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 trên `"RRLSLR"` (tự dựng; kiểm bằng mô phỏng ở dưới):

| i | ch | trước: pending_R, blocked | hành động | total |
|---|---|---|---|---|
| 0 | R | 0, F | pending_R = 1 | 0 |
| 1 | R | 1, F | pending_R = 2 | 0 |
| 2 | L | 2, F | R↔L (+2) và 1 R đâm đống đổ nát (+1) | 3 |
| 3 | S | 0, T | không có R chờ | 3 |
| 4 | L | 0, T | đâm vào vật cản (+1) | 4 |
| 5 | R | 0, T | pending_R = 1 (chạy thoát về phải) | 4 |

Đáp án `4`; công thức: bỏ `R` cuối ⇒ `"RRLSL"` có 4 xe chạy ✓.

In [14]:
def trace_counter(directions):
    total = pending = 0; blocked = False
    for i, ch in enumerate(directions):
        before = (pending, blocked)
        if ch == "R":
            pending += 1; act = "thêm một R chờ"
        elif ch == "S":
            act = f"{pending} R đâm vào S (+{pending})"; total += pending; pending = 0; blocked = True
        elif pending:
            act = f"R<->L (+2) và {pending - 1} R đâm đống đổ nát (+{pending - 1})"; total += pending + 1; pending = 0; blocked = True
        elif blocked:
            act = "L đâm vật cản (+1)"; total += 1
        else:
            act = "L thoát sang trái"
        print(f"i={i} ch={ch}: trước (pending={before[0]}, blocked={before[1]}) -> {act:42s} total={total}")
    return total

print("Đáp án:", trace_counter("RRLSLR"), "| mô phỏng:", collisions_simulate("RRLSLR"))
print()
print("Đáp án:", trace_counter("RLRSLL"))

i=0 ch=R: trước (pending=0, blocked=False) -> thêm một R chờ                             total=0
i=1 ch=R: trước (pending=1, blocked=False) -> thêm một R chờ                             total=0
i=2 ch=L: trước (pending=2, blocked=False) -> R<->L (+2) và 1 R đâm đống đổ nát (+1)     total=3
i=3 ch=S: trước (pending=0, blocked=True) -> 0 R đâm vào S (+0)                         total=3
i=4 ch=L: trước (pending=0, blocked=True) -> L đâm vật cản (+1)                         total=4
i=5 ch=R: trước (pending=0, blocked=True) -> thêm một R chờ                             total=4
Đáp án: 4 | mô phỏng: 4

i=0 ch=R: trước (pending=0, blocked=False) -> thêm một R chờ                             total=0
i=1 ch=L: trước (pending=1, blocked=False) -> R<->L (+2) và 0 R đâm đống đổ nát (+0)     total=2
i=2 ch=R: trước (pending=0, blocked=True) -> thêm một R chờ                             total=2
i=3 ch=S: trước (pending=1, blocked=True) -> 1 R đâm vào S (+1)                         total=3
i=4 ch=L: 

In [15]:
def trace_simulation(directions):
    n = len(directions); pos = [2 * i for i in range(n)]; state = list(directions); total = 0
    for step in range(1, 2 * n + 2):
        stopped_at = defaultdict(int)
        for i in range(n):
            if state[i] == "S": stopped_at[pos[i]] += 1
        for i in range(n):
            if state[i] == "R": pos[i] += 1
            elif state[i] == "L": pos[i] -= 1
        groups = defaultdict(list)
        for i in range(n): groups[pos[i]].append(i)
        events = []
        for p, ids in sorted(groups.items()):
            movers = [i for i in ids if state[i] != "S"]
            if not movers or len(ids) < 2: continue
            add = len(movers) if stopped_at[p] else (2 if {"R", "L"} <= {state[i] for i in movers} else 0)
            total += add; events.append(f"xe {ids} tại {p / 2} (+{add})")
            for i in ids: state[i] = "S"
        if events:
            print(f"t = {step / 2}: " + "; ".join(events) + f" -> tổng {total}")
    return total

print("Đáp án:", trace_simulation("RLRSLL"))

t = 0.5: xe [0, 1] tại 0.5 (+2) -> tổng 2
t = 1.0: xe [2, 3, 4] tại 3.0 (+2) -> tổng 4
t = 2.0: xe [2, 3, 4, 5] tại 3.0 (+1) -> tổng 5
Đáp án: 5


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def countCollisions(self, directions: str) -> int:
        # L ở đầu chạy thoát sang trái, R ở cuối chạy thoát sang phải;
        # mọi xe đang chạy khác đều bị dừng và góp đúng 1 va chạm.
        core = directions.lstrip("L").rstrip("R")
        return len(core) - core.count("S")

for d, exp in examples:
    assert Solution().countCollisions(d) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (mỗi xe bị dừng góp đúng 1).** Mỗi va chạm "chạy ↔ đứng yên" làm **một** xe chạy dừng lại và cộng 1. Mỗi va chạm "chạy ↔ chạy ngược chiều" làm **hai** xe chạy dừng lại và cộng 2. Xe đứng yên ban đầu không bao giờ "góp" (nó không làm tăng bộ đếm khi bị đâm — bộ đếm tăng theo xe chạy). Vậy tổng số va chạm = **số xe đang chạy mà cuối cùng bị dừng**.

**Bổ đề 2 (xe nào thoát).**
- Một xe `L` thoát ⇔ mọi xe bên trái nó đều là `L` (cùng tốc độ, cùng hướng, khoảng cách giữ nguyên, không ai cản). Ngược lại, gọi `j` là xe gần nhất bên trái không phải `L`. Mọi xe giữa `j` và xe của ta đều là `L`. Nếu `j` là `S` thì xe `L` ngay sau `j` đâm vào nó và dừng, tạo vật cản cho xe `L` tiếp theo, … dây chuyền tới xe của ta. Nếu `j` là `R` thì `j` và xe `L` ngay sau nó đâm nhau thành vật cản đứng yên, rồi dây chuyền tương tự. Vậy xe của ta chắc chắn bị dừng.
- Đối xứng: một xe `R` thoát ⇔ mọi xe bên phải nó là `R`.

Ghép lại: xe chạy thoát đúng là các `L` thuộc khối `L` đầu chuỗi và các `R` thuộc khối `R` cuối chuỗi; mọi xe chạy khác bị dừng. Theo Bổ đề 1, đáp án = số ký tự khác `S` sau khi bỏ hai khối đó.

Kiểm chứng máy: so công thức với mô phỏng vật lý trên **mọi** chuỗi độ dài ≤ 8 (mục 12).

**Kiểm chứng Bổ đề 1 theo từng xe.** Ta sửa mô phỏng để ghi lại **xe nào** bị dừng do chính nó đang chạy (xe đang chạy dừng lại ⇒ góp 1). Tổng số "xe chạy bị dừng" phải bằng tổng điểm va chạm, và tập xe không bị dừng phải đúng là khối `L` đầu + khối `R` cuối (cùng các xe `S`).

In [17]:
def stopped_moving_cars(directions):
    n = len(directions); pos = [2 * i for i in range(n)]; state = list(directions); stopped = set()
    for _ in range(2 * n + 2):
        for i in range(n):
            if state[i] == "R": pos[i] += 1
            elif state[i] == "L": pos[i] -= 1
        groups = defaultdict(list)
        for i in range(n): groups[pos[i]].append(i)
        for p, ids in groups.items():
            if len(ids) >= 2 and any(state[i] != "S" for i in ids):
                for i in ids:
                    if state[i] != "S": stopped.add(i)
                    state[i] = "S"
    return stopped

rng = random.Random(12)
for _ in range(500):
    d = "".join(rng.choice("LRS") for _ in range(rng.randint(1, 12)))
    st = stopped_moving_cars(d)
    lead = len(d) - len(d.lstrip("L")); trail = len(d) - len(d.rstrip("R"))
    escaped = {i for i in range(lead)} | {i for i in range(len(d) - trail, len(d))}
    moving = {i for i, c in enumerate(d) if c != "S"}
    assert len(st) == collisions_simulate(d) and st == moving - escaped
print("Mỗi xe chạy bị dừng góp đúng 1; xe thoát = khối L đầu + khối R cuối (500 chuỗi)")
print("ví dụ 1: xe bị dừng =", sorted(stopped_moving_cars("RLRSLL")))

Mỗi xe chạy bị dừng góp đúng 1; xe thoát = khối L đầu + khối R cuối (500 chuỗi)
ví dụ 1: xe bị dừng = [0, 1, 2, 4, 5]


## 11. Phân tích Time / Space

Gọi `n = len(directions)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng nửa bước | `O(n²)` | `O(n)` |
| 2. Ngăn xếp | `O(n)` | `O(n)` |
| 3. Hai biến | `O(n)` | `O(1)` |
| 4. `lstrip/rstrip` | `O(n)` | `O(n)` |
| 5. Hai con trỏ | `O(n)` | `O(1)` |

In [18]:
rng = random.Random(2211)
big = "".join(rng.choice("LRS") for _ in range(10**5))
for name, fn in [("stack", collisions_stack), ("counter", collisions_counter), ("strip", collisions_strip),
                 ("two_ptr", collisions_two_pointers), ("Solution", lambda d: Solution().countCollisions(d))]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:9s}: {r} ({time.perf_counter() - t:.4f}s)")

stack    : 66855 (0.0071s)
counter  : 66855 (0.0042s)
strip    : 66855 (0.0002s)
two_ptr  : 66855 (0.0031s)
Solution : 66855 (0.0003s)


## 12. Bộ test

- Ví dụ đề.
- Biên: một xe; toàn `S`; toàn `L`; toàn `R`; `"RL"` (+2), `"RRL"`, `"RLL"`, `"RSSL"`, `"LRSRL"`.
- **Vét cạn mọi chuỗi** độ dài `1..8` trên `{L, R, S}` (≈ 9800 chuỗi): mọi cách khớp mô phỏng vật lý.
- Ngẫu nhiên dài hơn (seed cố định).

In [19]:
fns = {"simulate": collisions_simulate, "stack": collisions_stack, "counter": collisions_counter,
       "strip": collisions_strip, "two_ptr": collisions_two_pointers,
       "Solution": lambda d: Solution().countCollisions(d), "SolutionCounter": lambda d: SolutionCounter().countCollisions(d)}
edge = [("S", 0), ("L", 0), ("R", 0), ("SSS", 0), ("LLL", 0), ("RRR", 0), ("RL", 2), ("RRL", 3), ("RLL", 3),
        ("RSSL", 2), ("LRSRL", 3), ("SRRRL", 4)]
for d, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(d) == exp, (name, d, fn(d))
cnt = 0
for L in range(1, 9):
    for tup in product("LRS", repeat=L):
        d = "".join(tup); want = collisions_simulate(d)
        for name, fn in fns.items():
            if name != "simulate":
                assert fn(d) == want, (name, d)
        cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} chuỗi: OK")

Ví dụ + biên + vét cạn 9840 chuỗi: OK


In [20]:
rng = random.Random(42)
for _ in range(200):
    d = "".join(rng.choice(rng.choice(["LRS", "LR", "RS", "LS"])) for _ in range(rng.randint(1, 60)))
    want = collisions_simulate(d)
    for name, fn in fns.items():
        assert fn(d) == want, (name, d)
print("200 test ngẫu nhiên: mọi cách khớp mô phỏng")

200 test ngẫu nhiên: mọi cách khớp mô phỏng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm các cặp kề nhau `"RL"`, `"RS"`, `"SL"`

Bỏ qua hiệu ứng **dây chuyền**: các `R` phía sau sẽ đâm vào đống đổ nát.

In [21]:
def wrong_adjacent_only(d):
    total = 0
    for a, b in zip(d, d[1:]):
        if a == "R" and b == "L": total += 2
        elif (a == "R" and b == "S") or (a == "S" and b == "L"): total += 1
    return total

print("sai:", wrong_adjacent_only("RRL"), "| đúng:", collisions_simulate("RRL"))
print("ví dụ 1:", wrong_adjacent_only("RLRSLL"), "vs", 5)

sai: 2 | đúng: 3
ví dụ 1: 4 vs 5


In [22]:
print("sửa:", Solution().countCollisions("RRL"), Solution().countCollisions("RLRSLL"))

sửa: 3 5


### Lỗi 2 — Đếm mọi xe đang chạy (quên các xe thoát)

In [23]:
def wrong_count_all_moving(d):
    return len(d) - d.count("S")                     # SAI: L đầu và R cuối không va chạm

print("sai:", wrong_count_all_moving("LLRR"), "| đúng:", 0)

sai: 4 | đúng: 0


In [24]:
print("sửa:", collisions_strip("LLRR"))

sửa: 0


### Lỗi 3 — Strip nhầm phía (`lstrip('R')`, `rstrip('L')`)

`R` ở **đầu** không thoát (nếu bên phải có vật cản), còn `L` ở **đầu** mới thoát.

In [25]:
def wrong_strip_side(d):
    core = d.lstrip("R").rstrip("L")                 # SAI
    return len(core) - core.count("S")

print("sai:", wrong_strip_side("RLRSLL"), "| đúng:", 5, "| 'LLRR' sai:", wrong_strip_side("LLRR"))

sai: 2 | đúng: 5 | 'LLRR' sai: 4


In [26]:
print("sửa:", Solution().countCollisions("RLRSLL"), Solution().countCollisions("LLRR"))

sửa: 5 0


### Lỗi 4 — Quên đánh dấu `blocked` sau va chạm `R↔L`

Đống đổ nát sau `R↔L` là vật cản đứng yên; các `L` theo sau phải đâm vào (+1). Nếu chỉ đặt `blocked` khi gặp `'S'` thì bỏ sót.

In [27]:
def wrong_no_block_after_RL(d):
    total = pending = 0; blocked = False
    for ch in d:
        if ch == "R": pending += 1
        elif ch == "S": total += pending; pending = 0; blocked = True
        elif pending: total += pending + 1; pending = 0          # SAI: quên blocked = True
        elif blocked: total += 1
    return total

print("sai:", wrong_no_block_after_RL("RLL"), "| đúng:", collisions_simulate("RLL"))

sai: 2 | đúng: 3


In [28]:
print("sửa:", collisions_counter("RLL"))

sửa: 3


### Lỗi 5 — Va chạm `R↔L` khi có nhiều `R` chờ: cộng `2·pending` hoặc chỉ `2`

Chỉ chiếc `R` **gần nhất** đâm trực diện với `L` (+2); các `R` còn lại đâm vào đống đổ nát (+1 mỗi chiếc) ⇒ tổng `pending + 1`.

In [29]:
def wrong_rl_score(d):
    total = pending = 0; blocked = False
    for ch in d:
        if ch == "R": pending += 1
        elif ch == "S": total += pending; pending = 0; blocked = True
        elif pending: total += 2 * pending; pending = 0; blocked = True   # SAI
        elif blocked: total += 1
    return total

print("sai:", wrong_rl_score("RRRL"), "| đúng:", collisions_simulate("RRRL"))

sai: 6 | đúng: 4


In [30]:
print("sửa (pending + 1):", collisions_counter("RRRL"))

sửa (pending + 1): 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài va chạm trên đường thẳng** (ô tô, tiểu hành tinh, kiến…) ⇒ thường có lời giải **ngăn xếp** quét một chiều, hoặc **công thức** dựa vào "ai thoát được".
- Khi điểm số "chia đều cho từng đối tượng bị ảnh hưởng" (2 điểm cho 2 xe, 1 điểm cho 1 xe) ⇒ đếm **đối tượng** thay vì đếm **sự kiện**.
- Mô phỏng thời gian liên tục: **nhân đôi tọa độ** để gặp nhau tại điểm nguyên.
- Luôn viết mô phỏng "vật lý" chậm để kiểm công thức — vét cạn mọi chuỗi ngắn bắt được hầu hết lỗi.

## 15. Biến thể của bài

1. **LeetCode 735 — Asteroid Collision:** khối lượng khác nhau, vật nhỏ bị phá hủy ⇒ ngăn xếp.
2. **LeetCode 2751 — Robot Collisions (Hard):** robot có máu, va chạm trừ máu ⇒ ngăn xếp theo vị trí đã sắp.
3. **LeetCode 853 — Car Fleet:** xe nhanh đuổi kịp xe chậm và nhập đoàn ⇒ sắp theo vị trí, so thời gian tới đích.
4. **Số xe cuối cùng còn chạy** (bài tập 5): đúng bằng số `L` đầu + số `R` cuối.
5. **Thời điểm va chạm cuối cùng:** cần mô phỏng hoặc phân tích khoảng cách — thử bằng hàm mô phỏng.

Minh họa biến thể 1 (Asteroid Collision, tự dựng — kiểm bằng mô phỏng ngây thơ lặp tới khi ổn định):

In [31]:
def asteroid_collision(ast):
    st = []
    for a in ast:
        alive = True
        while alive and a < 0 and st and st[-1] > 0:
            if st[-1] < -a: st.pop(); continue
            if st[-1] == -a: st.pop()
            alive = False
        if alive: st.append(a)
    return st

def asteroid_naive(ast):
    ast = list(ast)
    changed = True
    while changed:
        changed = False
        for i in range(len(ast) - 1):
            if ast[i] > 0 > ast[i + 1]:
                a, b = ast[i], -ast[i + 1]
                ast = ast[:i] + ([ast[i]] if a > b else [ast[i + 1]] if b > a else []) + ast[i + 2:]
                changed = True; break
    return ast

rng = random.Random(735)
for _ in range(400):
    ast = [rng.choice([-1, 1]) * rng.randint(1, 5) for _ in range(rng.randint(1, 9))]
    assert asteroid_collision(ast) == asteroid_naive(ast)
print("Asteroid Collision (ngăn xếp) khớp mô phỏng ngây thơ; [5,10,-5] ->", asteroid_collision([5, 10, -5]))

Asteroid Collision (ngăn xếp) khớp mô phỏng ngây thơ; [5,10,-5] -> [5, 10]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng nửa bước | làm đúng vật lý | `O(n²)` | `O(n)` | chuẩn kiểm tra |
| 2. Ngăn xếp | pop các R khi gặp vật cản | `O(n)` | `O(n)` | dễ mở rộng (735, 2751) |
| 3. Hai biến | `pending_R`, `blocked` | `O(n)` | `O(1)` | khi cần chi tiết từng va chạm |
| 4. `lstrip/rstrip` | bỏ xe thoát, đếm xe chạy | `O(n)` | `O(n)` | **nộp bài**, một dòng |
| 5. Hai con trỏ | như 4, không tạo chuỗi | `O(n)` | `O(1)` | ngôn ngữ không có strip |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `countCollisions("SSRSSRLLRSLLRSRSSRLRRRRLLRRLSSRR")`.
2. **Tính tay:** `"RRSLL"` — dùng bảng trace của Cách 3.
3. **Điền TODO:** `escaped_cars(directions)` trả về số xe chạy **thoát** được.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `collision_events(directions)` trả về **danh sách** `(t, vị trí, điểm)` của các va chạm theo thời gian (dùng tọa độ thật, `t` là số thực), và kiểm tổng điểm bằng `countCollisions`.

In [32]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def escaped_cars(directions):
    # TODO
    pass

In [33]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_count_collisions(directions):
    # TODO
    pass

In [34]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def collision_events(directions):
    # TODO: dựa trên mô phỏng nửa bước, ghi lại (t, vị trí, điểm)
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Không có `L` ở đầu (bắt đầu bằng `S`). (2) Bỏ `RR` ở cuối. (3) Đếm ký tự khác `S` phần còn lại.

**Bài 2.** (1) Hai `R` gặp `S`: +2. (2) Hai `L` đâm vào vật cản: +1, +1. (3) Tổng `4`.

**Bài 3.** (1) Đếm `L` liên tiếp ở đầu. (2) Đếm `R` liên tiếp ở cuối. (3) Cộng lại (chú ý chuỗi toàn `L` hoặc toàn `R` không đếm trùng vì một xe chỉ có một hướng).

**Bài 4.** (1) `pending`, `blocked`, `total`. (2) Ba nhánh `R`, `S`, `L`. (3) Sau `R↔L` đặt `blocked = True`.

**Bài 5.** (1) Dùng tọa độ nhân đôi, mỗi nửa bước là `t += 0,5`. (2) Vị trí thật = tọa độ / 2. (3) `sum(điểm)` phải bằng `countCollisions`.

## 19. Đáp án tham khảo

In [35]:
# Bài 1, 2
s1 = "SSRSSRLLRSLLRSRSSRLRRRRLLRRLSSRR"
assert Solution().countCollisions(s1) == collisions_simulate(s1)
assert Solution().countCollisions("RRSLL") == 4
print("Bài 1:", Solution().countCollisions(s1), "| Bài 2: 4")
trace_counter("RRSLL")

Bài 1: 20 | Bài 2: 4
i=0 ch=R: trước (pending=0, blocked=False) -> thêm một R chờ                             total=0
i=1 ch=R: trước (pending=1, blocked=False) -> thêm một R chờ                             total=0
i=2 ch=S: trước (pending=2, blocked=False) -> 2 R đâm vào S (+2)                         total=2
i=3 ch=L: trước (pending=0, blocked=True) -> L đâm vật cản (+1)                         total=3
i=4 ch=L: trước (pending=0, blocked=True) -> L đâm vật cản (+1)                         total=4


4

In [36]:
# Bài 3, 4
def escaped_cars_ans(directions):
    return (len(directions) - len(directions.lstrip("L"))) + (len(directions) - len(directions.rstrip("R")))

my_count_collisions_ans = collisions_counter
rng = random.Random(3)
for _ in range(300):
    d = "".join(rng.choice("LRS") for _ in range(rng.randint(1, 12)))
    moving = len(d) - d.count("S")
    assert escaped_cars_ans(d) + my_count_collisions_ans(d) == moving
    assert my_count_collisions_ans(d) == collisions_simulate(d)
print("Bài 3, 4: OK (xe thoát + va chạm = số xe chạy)")

Bài 3, 4: OK (xe thoát + va chạm = số xe chạy)


In [37]:
# Bài 5
def collision_events_ans(directions):
    n = len(directions); pos = [2 * i for i in range(n)]; state = list(directions); events = []
    for step in range(1, 2 * n + 2):
        stopped_at = defaultdict(int)
        for i in range(n):
            if state[i] == "S": stopped_at[pos[i]] += 1
        for i in range(n):
            if state[i] == "R": pos[i] += 1
            elif state[i] == "L": pos[i] -= 1
        groups = defaultdict(list)
        for i in range(n): groups[pos[i]].append(i)
        for p, ids in sorted(groups.items()):
            movers = [i for i in ids if state[i] != "S"]
            if not movers or len(ids) < 2: continue
            pts = len(movers) if stopped_at[p] else (2 if {"R", "L"} <= {state[i] for i in movers} else 0)
            events.append((step / 2, p / 2, pts))
            for i in ids: state[i] = "S"
    return events

ev = collision_events_ans("RLRSLL")
print("Bài 5:", ev)
rng = random.Random(5)
for _ in range(200):
    d = "".join(rng.choice("LRS") for _ in range(rng.randint(1, 15)))
    assert sum(p for _, _, p in collision_events_ans(d)) == Solution().countCollisions(d)
print("Tổng điểm theo sự kiện khớp countCollisions trên 200 chuỗi")

Bài 5: [(0.5, 0.5, 2), (1.0, 3.0, 2), (2.0, 3.0, 1)]
Tổng điểm theo sự kiện khớp countCollisions trên 200 chuỗi


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
core = directions.lstrip('L').rstrip('R')   # bỏ xe chạy thoát
return len(core) - core.count('S')          # mỗi xe chạy còn lại góp đúng 1
```

**Template quét (khi cần chi tiết):** `pending_R` (R đang chờ), `blocked` (có vật cản bên trái); `S`: `+pending`; `L`: có `pending` ⇒ `+pending+1`, không có mà `blocked` ⇒ `+1`.

In [38]:
def template(directions):
    core = directions.lstrip("L").rstrip("R")
    return len(core) - core.count("S")
print([template(d) for d, _ in examples])

[5, 0]


**Checklist:**

- [ ] `L` ở **đầu** và `R` ở **cuối** chạy thoát (không va chạm).
- [ ] Mỗi xe chạy bị dừng góp **đúng 1** (R↔L = 2 cho hai xe).
- [ ] Hiệu ứng dây chuyền: nhiều `R` dồn vào một vật cản đều tính.
- [ ] Sau `R↔L`, chỗ đó là vật cản đứng yên (`blocked = True`).
- [ ] Đã thử: toàn `L`, toàn `R`, toàn `S`, `"RL"`, `"RRL"`, `"RLL"`.